# Task 3: Heart Disease Prediction
## DevelopersHub Corporation — AI/ML Engineering Internship

---

### Problem Statement
Build a **binary classification model** to predict whether a patient is at risk of heart disease based on clinical health features.

### Goal
- Load and clean the Heart Disease UCI Dataset
- Perform thorough Exploratory Data Analysis (EDA)
- Train and compare **Logistic Regression** and **Decision Tree** classifiers
- Evaluate using accuracy, ROC-AUC, confusion matrix, and classification report
- Identify the most important features affecting heart disease risk

### Dataset
**Heart Disease UCI Dataset** — 303 patients, 13 clinical features, binary target (0 = no disease, 1 = disease)

---

## 1. Setup & Dependencies

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn --quiet
print("✅ Libraries installed.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, auc,
    ConfusionMatrixDisplay, RocCurveDisplay
)

plt.style.use('seaborn-v0_8-darkgrid')
COLORS = {'no_disease': '#2196F3', 'disease': '#F44336',
          'lr': '#FF9800', 'dt': '#4CAF50', 'neutral': '#9C27B0'}

print("✅ All imports successful.")

## 2. Load Dataset

The Heart Disease UCI dataset is hosted publicly on the UCI ML Repository. We load it directly via URL — no manual download needed.

In [ ]:
# Load directly from UCI repository
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

COLUMNS = [
    'age', 'sex', 'cp', 'trestbps', 'chol',
    'fbs', 'restecg', 'thalach', 'exang',
    'oldpeak', 'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(URL, header=None, names=COLUMNS, na_values='?')

# Binarize target: 0 = no disease, 1 = heart disease present
df['target'] = (df['target'] > 0).astype(int)

print(f"✅ Dataset loaded: {df.shape[0]} patients, {df.shape[1]} columns")
df.head()

### Feature Dictionary

| Feature | Description |
|---|---|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mmHg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment |
| `ca` | Number of major vessels coloured by fluoroscopy (0–3) |
| `thal` | Thalassemia type (3 = normal, 6 = fixed defect, 7 = reversible defect) |
| `target` | **0 = No disease, 1 = Heart disease** |

## 3. Data Cleaning & Preprocessing

In [ ]:
print("=" * 45)
print("DATASET OVERVIEW")
print("=" * 45)
print(df.info())

print("\nMissing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  None ✅")

print("\nDescriptive Statistics:")
df.describe().round(2)

In [ ]:
# Handle missing values in 'ca' and 'thal' (only 6 rows total)
print(f"Rows before cleaning: {len(df)}")

# Strategy: impute with median (robust to outliers for these clinical features)
for col in ['ca', 'thal']:
    median_val = df[col].median()
    n_missing = df[col].isnull().sum()
    df[col].fillna(median_val, inplace=True)
    if n_missing > 0:
        print(f"  Imputed {n_missing} missing values in '{col}' with median ({median_val})")

print(f"Rows after cleaning : {len(df)}")
print(f"Missing values remaining: {df.isnull().sum().sum()} ✅")

print(f"\nTarget Distribution:")
counts = df['target'].value_counts()
print(f"  No Disease (0): {counts[0]} ({counts[0]/len(df)*100:.1f}%)")
print(f"  Disease    (1): {counts[1]} ({counts[1]/len(df)*100:.1f}%)")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── Target Distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels = ['No Disease', 'Heart Disease']
clrs   = [COLORS['no_disease'], COLORS['disease']]
counts = df['target'].value_counts().sort_index()

axes[0].bar(labels, counts.values, color=clrs, edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Target Class Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Patients')

axes[1].pie(counts.values, labels=labels, colors=clrs, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Target Class Proportion', fontweight='bold')

plt.suptitle('Heart Disease Dataset — Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Key Clinical Features by Target ───────────────────────────────────────────
continuous = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(1, len(continuous), figsize=(17, 5))

for ax, feat in zip(axes, continuous):
    data_0 = df[df['target'] == 0][feat]
    data_1 = df[df['target'] == 1][feat]
    ax.hist(data_0, bins=20, alpha=0.7, color=COLORS['no_disease'], label='No Disease', edgecolor='white')
    ax.hist(data_1, bins=20, alpha=0.7, color=COLORS['disease'],    label='Disease',    edgecolor='white')
    ax.axvline(data_0.mean(), color=COLORS['no_disease'], linestyle='--', linewidth=1.5)
    ax.axvline(data_1.mean(), color=COLORS['disease'],    linestyle='--', linewidth=1.5)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Continuous Feature Distributions by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Categorical Features vs Target ────────────────────────────────────────────
cat_features = ['sex', 'cp', 'fbs', 'exang']
cat_labels = {
    'sex':   {0: 'Female', 1: 'Male'},
    'cp':    {0: 'Typical', 1: 'Atypical', 2: 'Non-anginal', 3: 'Asymptomatic'},
    'fbs':   {0: 'FBS ≤120', 1: 'FBS >120'},
    'exang': {0: 'No Angina', 1: 'Angina'}
}

fig, axes = plt.subplots(1, 4, figsize=(17, 5))

for ax, feat in zip(axes, cat_features):
    ct = pd.crosstab(df[feat], df['target'], normalize='index') * 100
    ct.index = [cat_labels[feat].get(i, i) for i in ct.index]
    ct.columns = ['No Disease', 'Disease']
    ct.plot(kind='bar', ax=ax, color=[COLORS['no_disease'], COLORS['disease']],
            edgecolor='white', width=0.6, rot=20)
    ax.set_title(feat.upper(), fontweight='bold')
    ax.set_ylabel('% of patients')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 100)

plt.suptitle('Disease Rate by Categorical Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_categorical_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 8))

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            mask=mask, ax=ax, linewidths=0.4,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (incl. Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
print("\nTop Features Correlated with Target (heart disease):")
target_corr = corr['target'].drop('target').sort_values(key=abs, ascending=False)
print(target_corr.round(3).to_string())

In [ ]:
# ── Box Plots — Outlier Detection ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(17, 5))

for ax, feat in zip(axes, continuous):
    bp = ax.boxplot(
        [df[df['target']==0][feat], df[df['target']==1][feat]],
        labels=['No Disease', 'Disease'],
        patch_artist=True,
        medianprops={'color': 'black', 'linewidth': 2}
    )
    bp['boxes'][0].set_facecolor(COLORS['no_disease'] + '99')
    bp['boxes'][1].set_facecolor(COLORS['disease'] + '99')
    ax.set_title(feat, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=15)

plt.suptitle('Box Plots — Outlier Detection by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Preparation & Train/Test Split

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

# Stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features — critical for Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set : {len(X_train)} samples ({y_train.mean()*100:.1f}% disease)")
print(f"Test set     : {len(X_test)} samples  ({y_test.mean()*100:.1f}% disease)")
print(f"\n✅ Stratification preserved class balance in both splits.")

## 6. Model Training

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_train_sc, y_train)
y_pred_lr   = lr.predict(X_test_sc)
y_prob_lr   = lr.predict_proba(X_test_sc)[:, 1]
print("✅ Logistic Regression trained.")

# ── Decision Tree ──────────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt   = dt.predict(X_test_sc)
y_prob_dt   = dt.predict_proba(X_test_sc)[:, 1]
print("✅ Decision Tree trained.")

## 7. Model Evaluation

In [ ]:
# ── Accuracy & Cross-Validation ────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def full_eval(name, model, X_tr, X_te, y_tr, y_te):
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='accuracy')
    test_acc  = accuracy_score(y_te, model.predict(X_te))
    return {
        'Model'         : name,
        'CV Acc (mean)' : cv_scores.mean(),
        'CV Acc (std)'  : cv_scores.std(),
        'Test Accuracy' : test_acc,
    }

results = pd.DataFrame([
    full_eval('Logistic Regression', lr, X_train_sc, X_test_sc, y_train, y_test),
    full_eval('Decision Tree',       dt, X_train_sc, X_test_sc, y_train, y_test),
])

print("📊 Model Evaluation Results")
print("=" * 60)
print(results.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── Classification Reports ─────────────────────────────────────────────────────
for name, preds in [('Logistic Regression', y_pred_lr), ('Decision Tree', y_pred_dt)]:
    print(f"\n{'─'*45}")
    print(f"Classification Report — {name}")
    print(f"{'─'*45}")
    print(classification_report(y_test, preds, target_names=['No Disease', 'Disease']))

In [ ]:
# ── Confusion Matrices ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, name, color in zip(
    axes,
    [y_pred_lr, y_pred_dt],
    ['Logistic Regression', 'Decision Tree'],
    [COLORS['lr'], COLORS['dt']]
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{name}\nAccuracy: {acc:.3f}', fontweight='bold', fontsize=11)

plt.suptitle('Confusion Matrices — Heart Disease Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nReading the confusion matrix:")
print("  Top-left  (TN): Correctly predicted NO disease")
print("  Top-right (FP): Predicted disease, actually healthy — False Alarm")
print("  Bot-left  (FN): Predicted healthy, actually has disease — Missed Case ⚠️")
print("  Bot-right (TP): Correctly predicted disease present")

In [ ]:
# ── ROC Curves ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for probs, name, color in [
    (y_prob_lr, 'Logistic Regression', COLORS['lr']),
    (y_prob_dt, 'Decision Tree',       COLORS['dt']),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random Classifier (AUC = 0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='grey')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Heart Disease Classification', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("AUC Interpretation:")
print("  0.5 = random guessing | 0.7–0.8 = acceptable | 0.8–0.9 = excellent | >0.9 = outstanding")

## 8. Feature Importance Analysis

In [ ]:
# ── Logistic Regression Coefficients ──────────────────────────────────────────
lr_importance = pd.Series(
    np.abs(lr.coef_[0]),
    index=X.columns
).sort_values(ascending=True)

# ── Decision Tree Feature Importances ─────────────────────────────────────────
dt_importance = pd.Series(
    dt.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, importances, name, color in zip(
    axes,
    [lr_importance, dt_importance],
    ['Logistic Regression |Coefficient|', 'Decision Tree Gini Importance'],
    [COLORS['lr'], COLORS['dt']]
):
    bars = ax.barh(importances.index, importances.values,
                   color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, importances.values):
        ax.text(val + importances.max()*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Importance Score')

plt.suptitle('Feature Importance — Heart Disease Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Decision Tree Visualisation ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt,
    feature_names=X.columns.tolist(),
    class_names=['No Disease', 'Disease'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=False,
    proportion=True,
)
ax.set_title('Decision Tree — Heart Disease Prediction (max_depth=4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree_plot.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Results & Key Insights

In [ ]:
lr_acc  = accuracy_score(y_test, y_pred_lr)
dt_acc  = accuracy_score(y_test, y_pred_dt)
lr_auc  = auc(*roc_curve(y_test, y_prob_lr)[:2])
dt_auc  = auc(*roc_curve(y_test, y_prob_dt)[:2])
best    = 'Logistic Regression' if lr_auc >= dt_auc else 'Decision Tree'

top_lr = lr_importance.sort_values(ascending=False).index[0]
top_dt = dt_importance.sort_values(ascending=False).index[0]

print(f"""
{'='*60}
RESULTS SUMMARY — Heart Disease Prediction
{'='*60}

Logistic Regression:
  Test Accuracy : {lr_acc:.4f} ({lr_acc*100:.1f}%)
  ROC-AUC       : {lr_auc:.4f}
  Top Feature   : {top_lr}

Decision Tree:
  Test Accuracy : {dt_acc:.4f} ({dt_acc*100:.1f}%)
  ROC-AUC       : {dt_auc:.4f}
  Top Feature   : {top_dt}

🏆 Best model by ROC-AUC: {best}
{'='*60}

KEY INSIGHTS:
1. 'cp' (chest pain type) and 'thalach' (max heart rate) are
   consistently the strongest predictors of heart disease.

2. 'exang' (exercise-induced angina) and 'oldpeak' (ST depression)
   are strong signals — both reflect the heart's response to stress.

3. Logistic Regression achieves strong AUC with a simpler,
   more interpretable model — valuable in clinical settings.

4. False Negatives (predicting 'no disease' when disease IS present)
   are more dangerous than False Positives in medical screening.
   Recall for the Disease class is the most critical metric here.

LIMITATIONS & NEXT STEPS:
- Dataset is small (303 patients) — larger datasets would improve
  generalisation significantly.
- Try ensemble methods: Random Forest, XGBoost, or SVM.
- Tune classification threshold to optimise Recall over Precision
  for a medical screening use case.
- Consider SHAP values for deeper explainability of predictions.
""")